# FinQA 数据资产准备

本 Notebook 为银行金融大模型量化项目准备三类资产：

- `calibration`：用于 AWQ/GPTQ/SmoothQuant 校准；
- `quality_dev` 和 `quality_test`：用于质量评估；
- `high_risk_regression`：固定的金融数值回归集。

数据会缓存到 Google Drive，避免 Colab 运行时重置后重复下载。

In [ ]:
!pip -q install -U datasets huggingface_hub pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 载入项目源码

如果你已经把项目目录上传到 Colab，请修改 `PROJECT_DIR`。如果项目放在 Git 仓库，可以填写 `REPO_URL` 自动克隆。

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/OptimizationSystem'
REPO_URL = ''

if REPO_URL and not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} 不存在，请上传项目或填写 REPO_URL。'
    )

sys.path.insert(0, PROJECT_DIR)

In [16]:
import importlib
import src.data.finqa_assets as finqa_assets

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
print('FinQA loader:', finqa_assets.__file__)

OUTPUT_DIR = (
    '/content/drive/MyDrive/'
    'banking_llm_project/datasets'
)

assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)

assets.summary()

FinQA loader: /content/drive/MyDrive/OptimizationSystem/src/data/finqa_assets.py


Saving the dataset (0/1 shards):   0%|          | 0/1024 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/883 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/91 [00:00<?, ? examples/s]

{'calibration': 1024,
 'quality_dev': 883,
 'quality_test': 1064,
 'high_risk_regression': 91,
 'manifest': {'finqa_dataset': 'czyssrs/FinQA',
  'verified_dataset': 'Aiera/finqa-verified',
  'finqa_revision': 'main',
  'verified_revision': 'default',
  'seed': 42,
  'calibration_size': 1024,
  'sizes': {'calibration': 1024,
   'quality_dev': 883,
   'quality_test': 1064,
   'high_risk_regression': 91},
  'schema_version': 1,
  'assets': {'calibration': {'path': 'calibration_finqa_1024',
    'version': 'main',
    'size': 1024,
    'sha256': '34af0e443492d4fdc831712c67cd6f12168e923b187b0175817ed9457bd6adfd',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_dev': {'path': 'quality_dev_finqa',
    'version': 'main',
    'size': 883,
    'sha256': '7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_test': {'path': 'quality_test_finqa',
    'vers

In [ ]:
print('校准集:', len(assets.calibration))
print('质量调优集:', len(assets.quality_dev))
print('最终质量评估集:', len(assets.quality_test))
print('高风险回归集:', len(assets.high_risk_regression))
print('第一条校准 Prompt:')
print(assets.calibration_prompts[0][:2000])

## 接入量化算法

`assets.calibration_prompts` 可以直接传给你的 AWQ/GPTQ 校准流程。不要把答案字段拼进校准 Prompt。

In [ ]:
calibration_prompts = assets.calibration_prompts

for prompt in calibration_prompts[:3]:
    print(prompt[:500])
    print('---')

In [ ]:
from src.data.finqa_assets import load_assets

reloaded_assets = load_assets(OUTPUT_DIR)
reloaded_assets.summary()

## 使用规则

1. 用 `calibration` 计算量化参数。
2. 用 `quality_dev` 选择 group size、敏感层和混合精度。
3. 配置冻结后才使用 `quality_test` 生成最终报告。
4. 每个 FP16、INT8、INT4、剪枝或后端版本都运行 `high_risk_regression`。
5. 重点统计“FP16 正确但压缩模型错误”的新增退化。

## TODO 4.1：模型输入校验（Colab）

下面先生成一个合成微型 GPT-2，验证 manifest、文件检查、SHA-256、FP16 加载和 Transformers 推理的完整链路。运行前请在 Colab 选择 **运行时 → 更改运行时类型 → T4 GPU**。

> 合成模型只用于验证校验工具，不能据此勾选真实项目的 Todo 4.1。正式验收时，请按照 `docs/MODEL_INPUTS.md` 将 manifest 指向上游交付的真实模型。

In [ ]:
%pip -q install -U transformers accelerate safetensors tokenizers

In [ ]:
import json
from pathlib import Path

import torch
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast

if not torch.cuda.is_available():
    raise RuntimeError('请先把 Colab 运行时切换为 T4 GPU，再重新运行此单元格。')

TEST_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_test')
MODEL_DIR = TEST_ROOT / 'synthetic_tiny_gpt2'
REPORT_DIR = Path('/content/drive/MyDrive/OptimizationSystem/out/model_input_test')
MANIFEST_PATH = TEST_ROOT / 'model_manifest.synthetic.json'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

vocabulary = {
    '[PAD]': 0, '[UNK]': 1, '[BOS]': 2, '[EOS]': 3,
    'hello': 4, 'bank': 5, 'finance': 6, 'test': 7,
}
tokenizer_backend = Tokenizer(WordLevel(vocabulary, unk_token='[UNK]'))
tokenizer_backend.pre_tokenizer = Whitespace()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token='[UNK]', pad_token='[PAD]',
    bos_token='[BOS]', eos_token='[EOS]',
)
tokenizer.save_pretrained(MODEL_DIR)

config = GPT2Config(
    vocab_size=len(vocabulary), n_positions=32, n_ctx=32,
    n_embd=32, n_layer=2, n_head=4,
    bos_token_id=2, eos_token_id=3, pad_token_id=0,
)
GPT2LMHeadModel(config).save_pretrained(MODEL_DIR, safe_serialization=True)
(MODEL_DIR / 'LICENSE_TEST_ONLY.txt').write_text(
    'Synthetic model generated locally for internal validator testing only.\n',
    encoding='utf-8',
)

weight_files = sorted(path.name for path in MODEL_DIR.glob('*.safetensors'))
tokenizer_files = [
    name for name in ('tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json')
    if (MODEL_DIR / name).is_file()
]
manifest = {
    'schema_version': 1,
    'model_id': 'local/synthetic-tiny-gpt2',
    'model_path': str(MODEL_DIR),
    'version': 'test-1.0',
    'commit': 'generated-locally',
    'weight_files': weight_files,
    'config_file': 'config.json',
    'tokenizer_files': tokenizer_files,
    'expected_sha256': {},
    'license': {
        'name': 'Synthetic test fixture',
        'file': 'LICENSE_TEST_ONLY.txt',
        'internal_use_approved': True,
        'usage_scope': 'Validator smoke testing only',
    },
    'runtime': {
        'framework': 'transformers',
        'model_class': 'causal_lm',
        'dtype': 'float16',
        'device': 'cuda:0',
        'trust_remote_code': False,
        'smoke_test_prompt': 'hello bank',
    },
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('模型目录:', MODEL_DIR)
print('Manifest:', MANIFEST_PATH)
print('权重文件:', weight_files)
print('Tokenizer 文件:', tokenizer_files)

In [ ]:
import importlib
import src.input_validation.model_input as model_input_validation

importlib.reload(model_input_validation)
static_report = model_input_validation.validate_model_input(MANIFEST_PATH)
static_payload = static_report.to_dict()
(REPORT_DIR / 'static.json').write_text(
    json.dumps(static_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(static_payload, ensure_ascii=False, indent=2))
assert static_report.validation_passed
assert not static_report.complete
print('静态检查通过；两项运行时检查尚未执行。')

In [ ]:
runtime_report = model_input_validation.validate_model_input(
    MANIFEST_PATH, runtime_check=True
)
runtime_payload = runtime_report.to_dict()
(REPORT_DIR / 'runtime.json').write_text(
    json.dumps(runtime_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(runtime_payload, ensure_ascii=False, indent=2))
assert runtime_report.complete, '存在未通过项，请查看上面的 detail。'
print('TODO 4.1 校验工具链冒烟测试：PASS')
print('报告目录:', REPORT_DIR)

### 使用真实模型验收

合成测试通过后，复制 `inputs/model_manifest.example.json`，将 `model_path`、权重分片、版本、commit、许可证和运行时配置替换为上游真实信息。许可证负责人确认前，不要把 `internal_use_approved` 设置为 `true`。真实模型生成的 `runtime.json` 六项全部为 `pass` 后，才能勾选 Todo 4.1。

## TODO 4.1：真实基础模型验收

本节默认使用 `Qwen/Qwen2.5-0.5B-Instruct` 完成真实模型验收。模型约 1 GB，可在 T4 GPU 上以 FP16 运行。代码会固定 Hugging Face commit、下载模型到 Google Drive、计算权重哈希并生成静态与运行时报告。

运行配置单元格前，请阅读模型卡和仓库中的许可证。只有你确认许可证允许当前内部使用范围后，才能把 `INTERNAL_USE_APPROVED` 改成 `True`。

In [ ]:
REAL_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REAL_MODEL_REVISION = 'main'
INTERNAL_USE_APPROVED = True  # 阅读并确认许可证后手动改为 True
INTERNAL_USAGE_SCOPE = 'Internal research, evaluation, and model optimization only'
REAL_TEST_PROMPT = '请用一句话说明什么是贷款利率。'

print('模型卡: https://huggingface.co/' + REAL_MODEL_ID)
print('内部使用许可已确认:', INTERNAL_USE_APPROVED)

In [ ]:
from huggingface_hub import HfApi, snapshot_download

REAL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
REAL_MODEL_DIR = REAL_ROOT / 'qwen2.5-0.5b-instruct'
REAL_REPORT_DIR = Path('/content/drive/MyDrive/OptimizationSystem/out/model_input_real')
REAL_MANIFEST_PATH = REAL_ROOT / 'model_manifest.json'
REAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
REAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

model_info = HfApi().model_info(REAL_MODEL_ID, revision=REAL_MODEL_REVISION)
REAL_MODEL_COMMIT = model_info.sha
reported_license = getattr(model_info.card_data, 'license', None) or 'not-declared'
print('解析后的 commit:', REAL_MODEL_COMMIT)
print('模型卡声明的许可证:', reported_license)

snapshot_download(
    repo_id=REAL_MODEL_ID,
    revision=REAL_MODEL_COMMIT,
    local_dir=REAL_MODEL_DIR,
    allow_patterns=[
        '*.json', '*.safetensors', '*.model', '*.tiktoken',
        'vocab.*', 'merges.*', 'LICENSE*',
    ],
)
print('真实模型目录:', REAL_MODEL_DIR)

In [ ]:
weight_files = sorted(
    str(path.relative_to(REAL_MODEL_DIR))
    for pattern in ('*.safetensors', 'pytorch_model*.bin')
    for path in REAL_MODEL_DIR.rglob(pattern)
    if path.is_file()
)
tokenizer_names = (
    'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model',
    'vocab.json', 'merges.txt', 'special_tokens_map.json',
)
tokenizer_files = [
    name for name in tokenizer_names if (REAL_MODEL_DIR / name).is_file()
]
license_files = sorted(
    path.name for path in REAL_MODEL_DIR.glob('LICENSE*') if path.is_file()
)
if not license_files:
    raise FileNotFoundError('模型仓库没有下载到 LICENSE 文件，不能完成许可证检查。')

real_manifest = {
    'schema_version': 1,
    'model_id': REAL_MODEL_ID,
    'model_path': str(REAL_MODEL_DIR),
    'version': 'Qwen2.5-0.5B-Instruct',
    'commit': REAL_MODEL_COMMIT,
    'weight_files': weight_files,
    'config_file': 'config.json',
    'tokenizer_files': tokenizer_files,
    'expected_sha256': {},
    'license': {
        'name': reported_license,
        'file': license_files[0],
        'internal_use_approved': INTERNAL_USE_APPROVED,
        'usage_scope': INTERNAL_USAGE_SCOPE,
    },
    'runtime': {
        'framework': 'transformers',
        'model_class': 'causal_lm',
        'dtype': 'float16',
        'device': 'cuda:0',
        'trust_remote_code': False,
        'smoke_test_prompt': REAL_TEST_PROMPT,
    },
}
REAL_MANIFEST_PATH.write_text(
    json.dumps(real_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Manifest:', REAL_MANIFEST_PATH)
print('权重文件:', weight_files)
print('Tokenizer 文件:', tokenizer_files)
print('许可证文件:', license_files[0])

In [ ]:
real_static_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH
)
real_static_payload = real_static_report.to_dict()
(REAL_REPORT_DIR / 'static.json').write_text(
    json.dumps(real_static_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_static_payload, ensure_ascii=False, indent=2))
assert real_static_report.validation_passed, (
    '静态检查未通过。若仅许可证项失败，请确认许可证后把 '
    'INTERNAL_USE_APPROVED 改为 True，并重新运行配置、manifest 和本单元格。'
)

In [ ]:
import gc

if not torch.cuda.is_available():
    raise RuntimeError('请先把 Colab 运行时切换为 T4 GPU。')
gc.collect()
torch.cuda.empty_cache()

real_runtime_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH, runtime_check=True
)
real_runtime_payload = real_runtime_report.to_dict()
(REAL_REPORT_DIR / 'runtime.json').write_text(
    json.dumps(real_runtime_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_runtime_payload, ensure_ascii=False, indent=2))
assert real_runtime_report.complete, '真实模型存在未通过项，请查看上面的 detail。'
print('TODO 4.1 真实模型验收：PASS')
print('报告目录:', REAL_REPORT_DIR)

### 完成条件

最后一个单元格输出 `TODO 4.1 真实模型验收：PASS`，且 `runtime.json` 中 `complete` 为 `true` 后，将 `model_manifest.json`、`static.json` 和 `runtime.json` 拉回项目，即可据此更新 Todo 4.1。

## TODO 4.2：评测资产输入验收

本节重新生成增强版数据 manifest，实际加载校准集、开发/最终评估集和高风险回归集，检查路径、版本、隔离、字段、指标接口、关键错误规则、样本数与 SHA-256。数据仍保存在原 `banking_llm_project/datasets` 路径，只有验收结果写入 `OptimizationSystem/out/dataset_input`。

In [ ]:
import importlib

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)
DATASET_MANIFEST_PATH = Path(OUTPUT_DIR) / 'manifest.json'
print('增强版数据 Manifest:', DATASET_MANIFEST_PATH)
print(json.dumps(assets.manifest, ensure_ascii=False, indent=2))

In [ ]:
import src.evaluation.finqa_metrics as finqa_metrics

importlib.reload(finqa_metrics)
metric_smoke = finqa_metrics.evaluate_numeric_answers(
    predictions=['$100', '12.5%'],
    references=['100', '0.125'],
)
regression_smoke = finqa_metrics.classify_numeric_regression(
    baseline_prediction='100',
    candidate_prediction='99',
    reference='100',
)
print('数值指标冒烟结果:', metric_smoke)
print('新增退化判定:', regression_smoke)
assert metric_smoke['numeric_accuracy'] == 1.0
assert regression_smoke['new_regression'] is True

In [ ]:
import shutil
import src.input_validation.dataset_input as dataset_input_validation

importlib.reload(dataset_input_validation)
dataset_report = dataset_input_validation.validate_dataset_input(
    DATASET_MANIFEST_PATH
)
dataset_payload = dataset_report.to_dict()
DATASET_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/dataset_input'
)
DATASET_RESULT_DIR.mkdir(parents=True, exist_ok=True)
(DATASET_RESULT_DIR / 'validation.json').write_text(
    json.dumps(dataset_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    DATASET_MANIFEST_PATH,
    DATASET_RESULT_DIR / 'dataset_manifest.json',
)
print(json.dumps(dataset_payload, ensure_ascii=False, indent=2))
assert dataset_report.complete, '评测资产存在未通过项，请查看上面的 detail。'
print('TODO 4.2 评测资产输入验收：PASS')
print('结果目录:', DATASET_RESULT_DIR)

### 4.2 完成条件

看到 `TODO 4.2 评测资产输入验收：PASS` 后，将 `out/dataset_input/dataset_manifest.json` 和 `out/dataset_input/validation.json` 拉回项目。确认 `validation.json` 中九项均为 `pass` 且 `complete` 为 `true`，即可勾选 Todo 4.2。

## TODO 4.3：硬件环境输入验收

本节采集 Colab 的真实 NVIDIA GPU、显存、驱动、CUDA、容器、PyTorch、Transformers 和拓扑信息。硬件 manifest 保存在原资产目录 `banking_llm_project/hardware`，项目结果保存在 `OptimizationSystem/out/hardware`。

> Colab 没有昇腾设备，因此本节只能完成 NVIDIA 部分。Todo 4.3 必须等真实昇腾机器的 manifest 合并通过后才能整体勾选。

In [ ]:
import importlib
import src.input_validation.hardware_input as hardware_input_validation

importlib.reload(hardware_input_validation)
HARDWARE_ASSET_DIR = Path(
    '/content/drive/MyDrive/banking_llm_project/hardware'
)
HARDWARE_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/hardware'
)
NVIDIA_MANIFEST_PATH = HARDWARE_ASSET_DIR / 'nvidia_colab.json'
HARDWARE_ASSET_DIR.mkdir(parents=True, exist_ok=True)
HARDWARE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

nvidia_manifest = hardware_input_validation.collect_nvidia_environment()
NVIDIA_MANIFEST_PATH.write_text(
    json.dumps(nvidia_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(nvidia_manifest, ensure_ascii=False, indent=2))
assert nvidia_manifest['accelerator']['available'], '没有检测到 NVIDIA GPU。'
print('NVIDIA 环境采集：PASS')
print('硬件 Manifest:', NVIDIA_MANIFEST_PATH)

In [ ]:
import shutil

nvidia_only_report = hardware_input_validation.validate_hardware_inputs(
    NVIDIA_MANIFEST_PATH
)
nvidia_only_payload = nvidia_only_report.to_dict()
(HARDWARE_RESULT_DIR / 'nvidia_validation.json').write_text(
    json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    NVIDIA_MANIFEST_PATH,
    HARDWARE_RESULT_DIR / 'nvidia_colab.json',
)
print(json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2))
assert nvidia_only_payload['checks'][0]['status'] == 'pass'
assert nvidia_only_payload['checks'][1]['status'] == 'pass'
assert nvidia_only_report.complete is False
print('NVIDIA 两项验收：PASS；昇腾与双平台合并验收：PENDING')
print('结果目录:', HARDWARE_RESULT_DIR)

### 昇腾环境下一步

在真实昇腾机器同步项目代码后执行：

```bash
python -m src.input_validation.hardware_input collect \
  --target ascend \
  --output /path/to/hardware/ascend.json
```

根据 `npu-smi` 和实际安装补齐芯片型号、数量、显存、CANN、驱动、固件、推理框架及通信字段，再与 `nvidia_colab.json` 执行合并验收。七项全部为 `pass` 后才能完成 Todo 4.3。

## TODO 5.2：FP16 基线推理服务

本节在真实 T4 上加载 Qwen2.5-0.5B-Instruct，执行单请求、两条样本批量推理和确定性重放，并把请求级性能日志、模型版本及运行环境写入 `OptimizationSystem/out/baseline`。模型和 FinQA 数据仍从原有路径读取，不写入该结果目录。

In [ ]:
import importlib
import shutil

import src.baseline.service as baseline_service

importlib.reload(baseline_service)
BASELINE_CONFIG_PATH = Path(PROJECT_DIR) / 'configs/baseline.json'
MODEL_MANIFEST_PATH = Path(
    '/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json'
)
if not MODEL_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f'找不到真实模型 manifest: {MODEL_MANIFEST_PATH}。'
        '请确认 Google Drive 已挂载，并先完成 TODO 4.1。'
    )
BASELINE_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/baseline'
)
BASELINE_LOG_PATH = BASELINE_RESULT_DIR / 'request_logs.jsonl'
BASELINE_RESULT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_LOG_PATH.unlink(missing_ok=True)

baseline = baseline_service.BaselineService.from_local_model(
    config_file=BASELINE_CONFIG_PATH,
    model_manifest_file=MODEL_MANIFEST_PATH,
    log_file=BASELINE_LOG_PATH,
)
print('FP16 基线模型加载：PASS')
print(json.dumps(baseline.model_metadata, ensure_ascii=False, indent=2))
print(json.dumps(baseline.environment_metadata(), ensure_ascii=False, indent=2))

In [ ]:
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)

single_request = baseline_service.InferenceRequest.from_finqa_row(
    assets.quality_dev[0],
    request_id='baseline-single-0001',
)
batch_requests = [
    baseline_service.InferenceRequest.from_finqa_row(
        assets.quality_dev[index],
        request_id=f'baseline-batch-{index + 1:04d}',
    )
    for index in range(2)
]
single_result = baseline.generate_one(single_request)
batch_results = baseline.generate_batch(batch_requests)
replay_result = baseline.generate_one(single_request)
replay_match = (
    single_result['output_text'] == replay_result['output_text']
)
assert len(batch_results) == 2
assert replay_match, '固定种子重放输出不一致。'
print('单请求输出:', single_result['output_text'])
print('批量输出:', [item['output_text'] for item in batch_results])
print('确定性重放一致:', replay_match)

In [ ]:
baseline_config = json.loads(
    BASELINE_CONFIG_PATH.read_text(encoding='utf-8')
)
log_records = [
    json.loads(line)
    for line in BASELINE_LOG_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
required_log_fields = {
    'request_id', 'success', 'batch_size', 'input_tokens',
    'output_tokens', 'latency_ms', 'output_tokens_per_second',
    'model_id', 'model_commit', 'dtype', 'device',
}
baseline_evidence = {
    'model_loaded': True,
    'config': baseline_config,
    'single_request': {
        'success': single_result['success'] is True,
        'request_id': single_result['request_id'],
        'output_text': single_result['output_text'],
    },
    'batch_request': {
        'success': len(batch_results) == 2,
        'result_count': len(batch_results),
        'request_ids': [item['request_id'] for item in batch_results],
    },
    'deterministic_replay_match': replay_match,
    'request_logs': {
        'path': str(BASELINE_LOG_PATH),
        'record_count': len(log_records),
        'all_success': all(row.get('success') is True for row in log_records),
        'required_fields_present': all(
            required_log_fields.issubset(row) for row in log_records
        ),
    },
    'model_metadata': baseline.model_metadata,
    'environment': baseline.environment_metadata(),
}
evidence_path = BASELINE_RESULT_DIR / 'baseline_evidence.json'
evidence_path.write_text(
    json.dumps(baseline_evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    BASELINE_CONFIG_PATH,
    BASELINE_RESULT_DIR / 'baseline_config.json',
)
baseline_report = baseline_service.validate_baseline_evidence(evidence_path)
baseline_payload = baseline_report.to_dict()
(BASELINE_RESULT_DIR / 'validation.json').write_text(
    json.dumps(baseline_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(baseline_payload, ensure_ascii=False, indent=2))
assert baseline_report.complete, '基线服务存在未通过项，请查看 detail。'
print('TODO 5.2 基线服务验收：PASS')
print('结果目录:', BASELINE_RESULT_DIR)

### 5.2 完成条件

看到 `TODO 5.2 基线服务验收：PASS` 后，将 `out/baseline` 目录拉回项目。确认 `validation.json` 中七项均为 `pass` 且 `complete` 为 `true`，即可勾选 Todo 5.2。

## TODO 5.3：统一 FP16 Benchmark

本节在同一 Tesla T4、同一 FP16 模型和统一生成方式下，测试 3 种 batch size、2 种输入长度和 2 种输出长度，共 12 个测试点。每个测试点预热 1 次、正式测量 5 次，并记录 TTFT、单 token 延迟、端到端 P50/P95/P99、tokens/s、requests/s、GPU 峰值显存和实际 KV Cache 张量显存。

In [ ]:
import importlib
import shutil

import src.baseline.service as baseline_service
import src.benchmark.runner as benchmark_runner
import src.data.finqa_assets as finqa_assets

importlib.reload(benchmark_runner)
BENCHMARK_CONFIG_PATH = Path(PROJECT_DIR) / 'configs/benchmark.json'
MODEL_MANIFEST_PATH = (
    Path(PROJECT_DIR) / 'out/model_input_real/model_manifest.json'
)
BENCHMARK_RESULT_DIR = Path(PROJECT_DIR) / 'out/benchmark'
BENCHMARK_RESULT_DIR.mkdir(parents=True, exist_ok=True)
benchmark_config = json.loads(
    BENCHMARK_CONFIG_PATH.read_text(encoding='utf-8')
)
if 'baseline' not in globals():
    baseline = baseline_service.BaselineService.from_local_model(
        config_file=Path(PROJECT_DIR) / 'configs/baseline.json',
        model_manifest_file=MODEL_MANIFEST_PATH,
        log_file=BENCHMARK_RESULT_DIR / 'service.jsonl',
    )
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)
benchmark_request = baseline_service.InferenceRequest.from_finqa_row(
    assets.quality_dev[0], request_id='benchmark-workload'
)
benchmark_prompt = benchmark_request.render_prompt()
(BENCHMARK_RESULT_DIR / 'benchmark_prompt.txt').write_text(
    benchmark_prompt, encoding='utf-8'
)
benchmark_cases = benchmark_runner.build_cases(benchmark_config)
print('测试点数量:', len(benchmark_cases))
print('总生成调用次数:', len(benchmark_cases) * (
    benchmark_config['warmup_runs'] + benchmark_config['measured_runs']
))
print('结果目录:', BENCHMARK_RESULT_DIR)

In [ ]:
if 'benchmark_report' not in globals():
    benchmark_report = benchmark_runner.run_benchmark(
        baseline, benchmark_prompt, benchmark_config
    )
else:
    print('复用当前 Colab 会话中已完成的 Benchmark 结果。')
benchmark_report_path = (
    BENCHMARK_RESULT_DIR / 'baseline_fp16_report.json'
)
benchmark_report_path.write_text(
    json.dumps(benchmark_report, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    BENCHMARK_CONFIG_PATH,
    BENCHMARK_RESULT_DIR / 'benchmark_config.json',
)
benchmark_validation = benchmark_runner.validate_benchmark_report(
    benchmark_report_path
)
benchmark_validation_payload = benchmark_validation.to_dict()
(BENCHMARK_RESULT_DIR / 'validation.json').write_text(
    json.dumps(benchmark_validation_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
benchmark_summary = [
    {
        'case_id': case['case_id'],
        'ttft_p50_ms': case['latency_ms']['ttft']['p50'],
        'e2e_p50_ms': case['latency_ms']['end_to_end']['p50'],
        'e2e_p95_ms': case['latency_ms']['end_to_end']['p95'],
        'e2e_p99_ms': case['latency_ms']['end_to_end']['p99'],
        'tokens_per_second': case['throughput']['mean_output_tokens_per_second'],
        'requests_per_second': case['throughput']['mean_requests_per_second'],
        'peak_allocated_mib': case['memory']['peak_allocated_mib'],
        'kv_cache_mib': case['memory']['kv_cache_mib'],
    }
    for case in benchmark_report['cases']
]
print(json.dumps(benchmark_summary, ensure_ascii=False, indent=2))
print(json.dumps(benchmark_validation_payload, ensure_ascii=False, indent=2))
assert benchmark_validation.complete, (
    'Benchmark 存在未通过项，请查看 validation.json。'
)
print('TODO 5.3 Benchmark 验收：PASS')
print('结果目录:', BENCHMARK_RESULT_DIR)

### 5.3 完成条件

看到 `TODO 5.3 Benchmark 验收：PASS` 后，将 `out/benchmark` 目录拉回项目。确认 `validation.json` 中十一项均为 `pass` 且 `complete` 为 `true`，即可勾选 Todo 5.3。

## TODO 5.4：环境锁定

本节锁定真实 Tesla T4 基线环境，包括 Python、PyTorch、Transformers、CUDA、cuDNN、驱动、Colab 容器版本及完整 `pip freeze`。昇腾/CANN 必须等真实昇腾机器采集，基础容器必须实际构建并保存镜像 digest，因此这三项继续标记为 `not_run`。

In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/OptimizationSystem"
MODULE_FILE = Path(PROJECT_DIR) / "src/environment_lock/ma.py"

print("模块文件存在:", MODULE_FILE.is_file())
assert MODULE_FILE.is_file(), MODULE_FILE

# 确保项目目录优先
sys.path = [path for path in sys.path if path != PROJECT_DIR]
sys.path.insert(0, PROJECT_DIR)

# 清除已经加载的旧 src 包
for module_name in list(sys.modules):
  if module_name == "src" or module_name.startswith("src."):
    del sys.modules[module_name]

importlib.invalidate_caches()

In [ ]:
import importlib
import shutil

import src.environment_lock.manager as environment_lock_manager

importlib.reload(environment_lock_manager)
ENVIRONMENT_RESULT_DIR = Path(PROJECT_DIR) / 'out/environment_lock'
ENVIRONMENT_RESULT_DIR.mkdir(parents=True, exist_ok=True)
NVIDIA_HARDWARE_PATH = (
    Path(PROJECT_DIR) / 'out/hardware/nvidia_colab.json'
)
BENCHMARK_REPORT_PATH = (
    Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
)
DEPENDENCY_LOCK_PATH = (
    ENVIRONMENT_RESULT_DIR / 'requirements.freeze.txt'
)
NVIDIA_LOCK_PATH = (
    ENVIRONMENT_RESULT_DIR / 'nvidia_environment.json'
)
ASCEND_PENDING_PATH = (
    ENVIRONMENT_RESULT_DIR / 'ascend_environment.pending.json'
)
environment_lock_manager.capture_pip_freeze(DEPENDENCY_LOCK_PATH)
nvidia_lock = environment_lock_manager.create_nvidia_lock(
    hardware_manifest_file=NVIDIA_HARDWARE_PATH,
    benchmark_report_file=BENCHMARK_REPORT_PATH,
    dependency_lock_file=DEPENDENCY_LOCK_PATH,
    output_file=NVIDIA_LOCK_PATH,
)
shutil.copy2(
    Path(PROJECT_DIR) / 'configs/environment/ascend.pending.json',
    ASCEND_PENDING_PATH,
)
print('NVIDIA 环境锁定文件:', NVIDIA_LOCK_PATH)
print('依赖包数量:', nvidia_lock['dependencies']['package_count'])
print('pip freeze SHA-256:', nvidia_lock['dependencies']['sha256'])
print(json.dumps({
    'runtime': nvidia_lock['runtime'],
    'software': nvidia_lock['software'],
    'frameworks': nvidia_lock['frameworks'],
}, ensure_ascii=False, indent=2))

In [ ]:
environment_report = environment_lock_manager.validate_environment_lock(
    nvidia_lock_file=NVIDIA_LOCK_PATH,
    dependency_lock_file=DEPENDENCY_LOCK_PATH,
    ascend_lock_file=ASCEND_PENDING_PATH,
)
environment_payload = environment_report.to_dict()
(ENVIRONMENT_RESULT_DIR / 'validation.json').write_text(
    json.dumps(environment_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
statuses = {
    check['name']: check['status']
    for check in environment_payload['checks']
}
expected_pass = {
    'nvidia_environment_file_created',
    'python_torch_transformers_versions_pinned',
    'inference_framework_version_pinned',
    'dependency_lock_saved',
    'hardware_and_driver_recorded',
}
expected_not_run = {
    'ascend_environment_file_created',
    'cuda_and_cann_versions_pinned',
    'base_container_image_built',
}
assert all(statuses[name] == 'pass' for name in expected_pass)
assert all(statuses[name] == 'not_run' for name in expected_not_run)
assert environment_report.complete is False
print(json.dumps(environment_payload, ensure_ascii=False, indent=2))
print('TODO 5.4 NVIDIA 环境锁定：PASS')
print('昇腾/CANN/基础容器构建：PENDING')
print('结果目录:', ENVIRONMENT_RESULT_DIR)

### 5.4 当前完成条件

看到 `TODO 5.4 NVIDIA 环境锁定：PASS` 后，将 `out/environment_lock` 目录拉回项目。预期五项为 `pass`、三项为 `not_run`；只能勾选已经有真实证据的子项，不能把 5.4 整体标记完成。

### 5.5 质量实验环境对齐

质量实验必须与已完成的 Benchmark 使用同一套 Transformers 版本。当前锁定版本为 `5.18.0`。如果下面单元格提示版本不一致，请先安装指定版本并重启 Colab 运行时，再重新运行前置导入、基线服务和质量实验单元。

In [ ]:
import importlib.metadata
EXPECTED_TRANSFORMERS_VERSION = '5.18.0'
actual_transformers_version = importlib.metadata.version('transformers')
print('当前 Transformers:', actual_transformers_version)
if actual_transformers_version != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f'版本不一致：需要 {EXPECTED_TRANSFORMERS_VERSION}，'
        f'当前为 {actual_transformers_version}。请先运行：\n'
        f'%pip install -q --upgrade transformers=={EXPECTED_TRANSFORMERS_VERSION}\n'
        '然后重启 Colab 运行时，并重新运行项目导入、5.2 和 5.3 的前置单元。'
    )
print('Benchmark/质量实验 Transformers 版本对齐：PASS')

## TODO 5.5：FP16 质量基线与 NVIDIA 阶段验收

本节在版本化 FinQA development 集的全部 883 条样本上运行 FP16 模型，保存逐样本预测并计算数值准确率和解析率。该结果作为后续量化与剪枝方案的质量参照；最终 test 集继续保留，不在本阶段使用。

In [17]:
import json
from pathlib import Path
import importlib
import shutil

import src.baseline.service as baseline_service
import src.data.finqa_assets as finqa_assets
import src.evaluation.baseline_quality as baseline_quality
import src.evaluation.stage5_acceptance as stage5_acceptance

importlib.reload(baseline_quality)
importlib.reload(stage5_acceptance)
QUALITY_CONFIG_PATH = (
    Path(PROJECT_DIR) / 'configs/baseline_quality.json'
)
QUALITY_RESULT_DIR = Path(PROJECT_DIR) / 'out/baseline_quality_experiment'
STAGE5_RESULT_DIR = Path(PROJECT_DIR) / 'out/stage5'
QUALITY_RESULT_DIR.mkdir(parents=True, exist_ok=True)
STAGE5_RESULT_DIR.mkdir(parents=True, exist_ok=True)
quality_config = json.loads(
    QUALITY_CONFIG_PATH.read_text(encoding='utf-8')
)
quality_log_path = QUALITY_RESULT_DIR / 'quality_service.jsonl'
quality_log_path.unlink(missing_ok=True)
if 'baseline' in globals():
    quality_service = baseline_service.BaselineService(
        model=baseline.model,
        tokenizer=baseline.tokenizer,
        torch_module=baseline.torch,
        config=baseline.config,
        model_metadata=baseline.model_metadata,
        log_file=quality_log_path,
    )
else:
    quality_service = baseline_service.BaselineService.from_local_model(
        config_file=Path(PROJECT_DIR) / 'configs/baseline.json',
        model_manifest_file=(
            Path(PROJECT_DIR) / 'out/model_input_real/model_manifest.json'
        ),
        log_file=quality_log_path,
    )
if 'assets' not in globals():
    assets = finqa_assets.load_assets(OUTPUT_DIR)
print('评测样本数:', len(assets.quality_dev))
print('batch size:', quality_config['batch_size'])
print('质量结果目录:', QUALITY_RESULT_DIR)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

评测样本数: 883
batch size: 4
质量结果目录: /content/drive/MyDrive/OptimizationSystem/out/baseline_quality_experiment


In [18]:
import json
from pathlib import Path
quality_report = baseline_quality.run_baseline_quality(
    service=quality_service,
    dataset=assets.quality_dev,
    dataset_manifest=assets.manifest,
    quality_config=quality_config,
    output_dir=QUALITY_RESULT_DIR,
)
quality_report_path = (
    QUALITY_RESULT_DIR / 'baseline_quality_report.json'
)
quality_report_path.write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
shutil.copy2(
    QUALITY_CONFIG_PATH, QUALITY_RESULT_DIR / 'baseline_quality_config.json'
)
quality_validation = baseline_quality.validate_quality_report(
    quality_report_path
)
quality_validation_payload = quality_validation.to_dict()
(QUALITY_RESULT_DIR / 'validation.json').write_text(
    json.dumps(quality_validation_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(quality_report['metrics'], ensure_ascii=False, indent=2))
print(json.dumps(quality_validation_payload, ensure_ascii=False, indent=2))
assert quality_validation.complete, (
    'FP16 质量基线存在未通过项，请拉回报告后分析。'
)
print('FP16 全量开发集质量基线：PASS')

Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 136/883
Quality evaluation: 140/883
Quality evaluation: 144/883
Quality evaluation

In [23]:
import json
from pathlib import Path
stage5_report = stage5_acceptance.validate_stage5(
    model_report_file=(
        Path(PROJECT_DIR) / 'out/model_input_real/reports/runtime.json'
    ),
    baseline_validation_file=(
        Path(PROJECT_DIR) / 'out/baseline/validation.json'
    ),
    benchmark_validation_file=(
        Path(PROJECT_DIR) / 'out/benchmark/validation.json'
    ),
    benchmark_report_file=(
        Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
    ),
    quality_validation_file=(
        QUALITY_RESULT_DIR / 'validation.json'
    ),
    environment_validation_file=(
        Path(PROJECT_DIR) / 'out/environment_lock/validation.json'
    ),
    dataset_validation_file=(
        Path(PROJECT_DIR) / 'out/dataset_input/validation.json'
    ),
    business_validation_file=(
        Path(PROJECT_DIR) / 'out/business_scope/validation.json'
    ),
    deliverable_files={
        'baseline_model_run_instructions': (
            Path(PROJECT_DIR) / 'docs/BASELINE_SERVICE.md'
        ),
        'baseline_quality_report': quality_report_path,
        'baseline_performance_report': (
            Path(PROJECT_DIR) / 'out/benchmark/baseline_fp16_report.json'
        ),
        'input_interface_document': (
            Path(PROJECT_DIR) / 'docs/INPUT_INTERFACE.md'
        ),
        'environment_lock_file': (
            Path(PROJECT_DIR) / 'out/environment_lock/nvidia_environment.json'
        ),
    },
)
stage5_payload = stage5_report.to_dict()
(STAGE5_RESULT_DIR / 'validation.json').write_text(
    json.dumps(stage5_payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(stage5_payload, ensure_ascii=False, indent=2))
assert stage5_report.complete, '第 5 章 NVIDIA 阶段验收未通过。'
print('TODO 5.5 NVIDIA 基线阶段验收：PASS')
print('质量结果目录:', QUALITY_RESULT_DIR)
print('阶段验收目录:', STAGE5_RESULT_DIR)

{
  "scope": "nvidia_t4_baseline",
  "complete": true,
  "checks": [
    {
      "name": "fp16_or_bf16_model_stable",
      "status": "pass",
      "detail": "Model loading and repeated baseline inference passed."
    },
    {
      "name": "benchmark_repeatable",
      "status": "pass",
      "detail": "The versioned benchmark contains repeated measurements."
    },
    {
      "name": "baseline_business_quality_confirmed",
      "status": "pass",
      "detail": "The complete FinQA development baseline passed quality validation."
    },
    {
      "name": "baseline_performance_confirmed",
      "status": "pass",
      "detail": "The FP16 latency, throughput, and memory report passed validation."
    },
    {
      "name": "nvidia_inputs_and_environment_versioned",
      "status": "pass",
      "detail": "NVIDIA-stage inputs and environment versions are registered."
    }
  ],
  "deliverables": {
    "baseline_model_run_instructions": true,
    "baseline_quality_report": true,
    "b

### 第 5 章当前完成条件

看到 `TODO 5.5 NVIDIA 基线阶段验收：PASS` 后，将 `out/baseline_quality` 和 `out/stage5` 两个目录拉回项目。确认质量验收五项、阶段验收五项及交付物全部通过后，可完成 5.5；5.4 中昇腾/CANN/容器三项仍独立保持未完成。